# ECRTM PPD Notebook

Notebook unifie pour entrainer ECRTM sur les memes datasets que les autres modeles et exporter des `.mat` compatibles.


## 1) Install (Kaggle-safe)


In [ ]:
!pip install -q --upgrade-strategy only-if-needed scipy pyyaml rich


## 2) Imports


In [ ]:
import os
import sys
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import yaml
from sklearn import metrics
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 3) Dataset and config discovery


In [ ]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz',
    'test_bow.npz',
    'train_labels.txt',
    'test_labels.txt',
    'vocab.txt',
    'word_embeddings.npz',
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_roots = [
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data',
        PROJECT_ROOT / 'ECTRM_source' / 'data',
    ]

    for root in local_roots:
        for ds in TARGET_DATASETS:
            p = root / ds
            if has_required_files(p):
                found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand

    return found


def discover_cfg_paths():
    cfg = {}

    local_cfg_roots = [
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'ECRTM' / 'configs' / 'model',
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'configs' / 'model',
    ]

    for root in local_cfg_roots:
        for ds in TARGET_DATASETS:
            p = root / f'ECRTM_{ds}.yaml'
            if p.exists():
                cfg[ds] = p

    scan_roots = [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]
    for ds in TARGET_DATASETS:
        if ds in cfg:
            continue
        name = f'ECRTM_{ds}.yaml'
        for scan_root in scan_roots:
            if not scan_root.exists():
                continue
            found = None
            for r, _, files in os.walk(scan_root):
                if name in files:
                    found = Path(r) / name
                    break
            if found is not None:
                cfg[ds] = found
                break

    return cfg


DATASET_DIRS = discover_dataset_dirs()
CONFIGS = discover_cfg_paths()

print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)
print('\\nResolved datasets:')
for ds in TARGET_DATASETS:
    print(' -', ds, DATASET_DIRS.get(ds))
print('\\nResolved configs:')
for ds in TARGET_DATASETS:
    p = CONFIGS.get(ds)
    print(' -', ds, p, 'exists=', (p is not None and p.exists()))


## 4) Load official ECRTM model


In [ ]:
def looks_like_ecrtm_root(path: Path) -> bool:
    return (
        path.exists()
        and (path / 'models' / 'ECRTM.py').exists()
        and (path / 'models' / 'ECR.py').exists()
    )


CANDIDATE_ECRTM_ROOTS = [
    PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
    PROJECT_ROOT / 'ECTRM_source' / 'ECRTM',
]

ECRTM_ROOT = None
for cand in CANDIDATE_ECRTM_ROOTS:
    if looks_like_ecrtm_root(cand):
        ECRTM_ROOT = cand
        break

if ECRTM_ROOT is None:
    scan_roots = [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]
    for scan_root in scan_roots:
        if not scan_root.exists():
            continue
        for r, _, files in os.walk(scan_root):
            rp = Path(r)
            if 'ECRTM.py' in files and rp.name == 'models':
                cand = rp.parent
                if looks_like_ecrtm_root(cand):
                    ECRTM_ROOT = cand
                    break
        if ECRTM_ROOT is not None:
            break

if ECRTM_ROOT is None:
    raise FileNotFoundError('Impossible de trouver le dossier ECRTM avec models/ECRTM.py')

sys.path.insert(0, str(ECRTM_ROOT))

from models.ECRTM import ECRTM
from models.ECR import ECR

print('ECRTM_ROOT:', ECRTM_ROOT)


## 5) Train / export helpers


In [ ]:
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'ECRTM') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'ECRTM')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_cfg(path):
    if path is None or not Path(path).exists():
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr.astype(np.float32)
    except Exception:
        data = np.load(path)
        if isinstance(data, np.lib.npyio.NpzFile):
            arr = data[list(data.keys())[0]]
        else:
            arr = data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None:
            if arr.ndim == 1:
                emb_dim = arr.size // vocab_size
                arr = arr[: vocab_size * emb_dim].reshape(vocab_size, emb_dim)
            elif arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
                arr = arr.T
        return arr.astype(np.float32)


def build_args_from_cfg(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)
    cfg.setdefault('epochs', 500)
    cfg.setdefault('batch_size', 200)
    cfg.setdefault('learning_rate', 0.002)
    cfg.setdefault('lr_step_size', 125)
    cfg.setdefault('lr_scheduler', True)
    cfg.setdefault('dropout', 0.0)
    cfg.setdefault('en1_units', 200)
    cfg.setdefault('beta_temp', 0.2)
    cfg.setdefault('sinkhorn_alpha', 20)
    cfg.setdefault('OT_max_iter', 1000)
    cfg.setdefault('weight_loss_ECR', 100)

    cfg['num_topic'] = int(K)
    cfg['vocab_size'] = int(vocab_size)
    cfg['word_embeddings'] = word_emb
    return SimpleNamespace(**cfg)


def infer_theta(model, bow_csr, device, batch_size=256):
    model.eval()
    out = []
    N = bow_csr.shape[0]
    with torch.no_grad():
        for start in range(0, N, batch_size):
            idx = np.arange(start, min(start + batch_size, N))
            bow = torch.from_numpy(bow_csr[idx].toarray().astype(np.float32)).to(device)
            theta = model.get_theta(bow)
            out.append(theta.cpu().numpy().astype(np.float32))
    return np.vstack(out)


def train_one_ecrtm(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} introuvable. Disponibles: {list(DATASET_DIRS.keys())}')

    set_seed(seed)
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))

    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    V = train_bow.shape[1]

    word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=V)
    if word_emb.shape[0] != V:
        raise ValueError(f'word_embeddings shape mismatch: {word_emb.shape} vs vocab_size={V}')

    args = build_args_from_cfg(cfg, K=K, vocab_size=V, word_emb=word_emb)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ECRTM(args).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(args.learning_rate))
    scheduler = None
    if getattr(args, 'lr_scheduler', False):
        scheduler = StepLR(optimizer, step_size=int(args.lr_step_size), gamma=0.5)

    X = torch.from_numpy(train_bow.toarray().astype(np.float32))
    loader = DataLoader(TensorDataset(X), batch_size=int(args.batch_size), shuffle=True)

    for epoch in range(1, int(args.epochs) + 1):
        model.train()
        losses = []

        for (bow,) in loader:
            bow = bow.to(device)
            rst = model(bow)
            loss = rst['loss']

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(float(loss.detach().cpu().item()))

        if scheduler is not None:
            scheduler.step()

        if epoch == 1 or epoch % 10 == 0:
            print(f"{dataset} K={K} epoch {epoch:03d}/{args.epochs} loss={np.mean(losses):.4f}")

    model.eval()
    with torch.no_grad():
        beta = model.get_beta().detach().cpu().numpy().astype(np.float32)

    train_theta = infer_theta(model, train_bow, device=device, batch_size=256)
    test_theta = infer_theta(model, test_bow, device=device, batch_size=256)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_ECRTM_K{K}_seed{seed}.mat'

    scipy.io.savemat(
        str(out_path),
        {
            'beta': beta,
            'train_theta': train_theta,
            'test_theta': test_theta,
            'traintheta': train_theta,
            'testtheta': test_theta,
        },
    )

    print('Saved:', out_path)
    print('beta', beta.shape, 'train_theta', train_theta.shape, 'test_theta', test_theta.shape)
    return out_path


## 6) Run


In [ ]:
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        train_one_ecrtm(dataset, K=K, seed=RANDOM_SEED)


## 7) Metrics and topics files


In [ ]:
import pandas as pd


def topic_diversity_from_beta(beta, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


rows = []
TOPN = 15
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    ds_out = OUTPUT_DIR / dataset
    vocab = load_vocab(DATASET_DIRS[dataset] / 'vocab.txt')
    labels = np.loadtxt(DATASET_DIRS[dataset] / 'test_labels.txt', dtype=np.int32)

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_ECRTM_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            'model': 'ECRTM',
            'dataset': dataset,
            'K': K,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top15': float(topic_diversity_from_beta(beta, topn=15)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_ECRTM_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print('Saved:', txt_path)

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / 'ECRTM_metrics_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)
else:
    print('No ECRTM .mat found yet')

print('\\nFiles in OUTPUT_DIR:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print('-', p.relative_to(OUTPUT_DIR))
